# PaperMind v4 — Drive parquet → Supabase static facts upload

**Kapsam (envanter §3 + §10.1 — A-evidence yollar):**
1. `~/Dataleak/facts/dim_theme.parquet` — 4,516 × 11
2. `~/Dataleak/dims/dim_theme_embedding.parquet` — 4,516 × 4 (256-d vector)
3. `~/Dataleak/facts/fact_theme_year_aggregates.parquet` — 53,943 × 15 (t-ESTRA 7/8)
4. `~/Dataleak/facts/fact_gap_matrix.parquet` — 504,436 × 12 (M1+M7+M8)

**Önkoşullar:** Migration 0001 (schema_v1) APPLIED ✓ ; Migration 0002 (pgvector + 3 tablo) → Cell 3'te uygulanır.

**Disiplin:** Run All YASAK — hücre hücre koş (1+3). Cell 2 schema audit FAIL ise INSERT_COLS güncelle, devam etme.

**Drive konumu:** `/content/drive/MyDrive/Dataleak/...` (Drive senkronlu Mac `~/Dataleak/` ile aynı kök; ENVANTER §1.1).

## Cell 1 — Setup (Drive mount + install + DB connection + paths)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

!pip -q install psycopg2-binary==2.9.9 pandas==2.2.2 pyarrow==17.0.0

import os, json, time, math
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

# Colab Secrets (sol panel gear ikon) → SUPABASE_DB_URL ekle
# Değer: postgresql://postgres.<ref>:<pwd>@aws-1-eu-central-1.pooler.supabase.com:5432/postgres
DB_URL = userdata.get('SUPABASE_DB_URL')
assert DB_URL and DB_URL.startswith('postgresql://'), 'Colab Secrets → SUPABASE_DB_URL ekle (Session Pooler)'

# Drive yolları (ENVANTER.md §3 + §10.1 A-evidence)
DRIVE_ROOT = '/content/drive/MyDrive/Dataleak'
PATHS = {
    'dim_theme':                  f'{DRIVE_ROOT}/facts/dim_theme.parquet',
    'dim_theme_embedding':        f'{DRIVE_ROOT}/dims/dim_theme_embedding.parquet',
    'fact_theme_year_aggregates': f'{DRIVE_ROOT}/facts/fact_theme_year_aggregates.parquet',
    'fact_gap_matrix':            f'{DRIVE_ROOT}/facts/fact_gap_matrix.parquet',
}

CHUNK = 50_000  # bulk insert chunk size (Free tier WAL throughput optimum)

for name, path in PATHS.items():
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗'}  {name:30s} {path}")

# DB connection smoke test
with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute('SELECT version();')
    print('\nDB:', cur.fetchone()[0][:60])
    cur.execute("SELECT version FROM public.schema_migrations ORDER BY applied_at;")
    print('Applied migrations:', [r[0] for r in cur.fetchall()])

## Cell 2 — Schema audit (INSERT öncesi denetim — kolonlar envanterle uyumlu mu?)

Beklenen output (envanter §3+§10.1):
- `dim_theme`: 4,516 satır, 11 kolon (theme_id, label_en, description, keywords, subfield_id, subfield_name, field_id, field_name, domain_id, domain_name, works_count)
- `dim_theme_embedding`: 4,516 × 4 (theme_id, embedding[256], text_source, n_terms_used)
- `fact_theme_year_aggregates`: 53,943 × 15 (theme_id, year, paper_count, R_mean, R_std, E_mean, E_std, ED_k_simpson, ES_k_top10, TS_penalty, blocked_metrics, n_blocked, C_k, CD_k, MQ_k)
- `fact_gap_matrix`: 504,436 × 12 (matrix_id, axis_x, axis_y, depth, depth_norm, neighbor, E, feasibility, gap_value, publishability, matrix_confidence_prior, matrix_confidence_calibrated)

**FAIL koşulu:** kolon sayısı veya isimleri sapıyorsa Cell 4-7'de mapping güncelle, ASLA devam etme.

In [ ]:
for name, path in PATHS.items():
    print(f'\n=== {name} ===')
    if not os.path.exists(path):
        print(f'  ✗ DOSYA YOK: {path}')
        continue
    df = pd.read_parquet(path)
    print(f'  rows : {len(df):,}')
    print(f'  cols : {len(df.columns)}  →  {list(df.columns)}')
    print(f'  dtypes:')
    for c, dt in df.dtypes.items():
        print(f'    {c:30s} {dt}')
    print(f'  head(2):')
    print(df.head(2).to_string(max_colwidth=60))

## Cell 3 — Migration 0002 uygula (pgvector + 3 tablo DDL)

Idempotent: `IF NOT EXISTS` + `ON CONFLICT DO NOTHING`. Schema warehouse parquet ile birebir aynalı (B42-039 / L-013 manifest > planning hiyerarşisi).

In [ ]:
MIGRATION_0002 = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS public.dim_theme_embedding (
  theme_id      text PRIMARY KEY REFERENCES public.dim_theme(theme_id) ON DELETE CASCADE,
  embedding     vector(256) NOT NULL,
  text_source   text,
  n_terms_used  int,
  created_at    timestamptz NOT NULL DEFAULT now()
);
CREATE INDEX IF NOT EXISTS idx_theme_emb_hnsw ON public.dim_theme_embedding
  USING hnsw (embedding vector_cosine_ops) WITH (m=16, ef_construction=64);
ALTER TABLE public.dim_theme_embedding ENABLE ROW LEVEL SECURITY;
DROP POLICY IF EXISTS theme_emb_read_all ON public.dim_theme_embedding;
CREATE POLICY theme_emb_read_all ON public.dim_theme_embedding
  FOR SELECT TO authenticated USING (true);
DROP POLICY IF EXISTS theme_emb_write_service ON public.dim_theme_embedding;
CREATE POLICY theme_emb_write_service ON public.dim_theme_embedding
  FOR ALL TO service_role USING (true) WITH CHECK (true);

CREATE TABLE IF NOT EXISTS public.fact_theme_year_aggregates (
  theme_id        text NOT NULL REFERENCES public.dim_theme(theme_id) ON DELETE CASCADE,
  year            int NOT NULL,
  paper_count     int NOT NULL DEFAULT 0,
  r_mean          real,
  r_std           real,
  e_mean          real,
  e_std           real,
  ed_k_simpson    real,
  es_k_top10      real,
  ts_penalty      real,
  c_k             real,
  cd_k            real,
  mq_k            real,
  blocked_metrics text[] DEFAULT '{}',
  n_blocked       smallint DEFAULT 0,
  PRIMARY KEY (theme_id, year)
);
CREATE INDEX IF NOT EXISTS idx_theme_year_year ON public.fact_theme_year_aggregates(year);
CREATE INDEX IF NOT EXISTS idx_theme_year_mq   ON public.fact_theme_year_aggregates(mq_k DESC) WHERE mq_k IS NOT NULL;
CREATE INDEX IF NOT EXISTS idx_theme_year_e    ON public.fact_theme_year_aggregates(e_mean DESC) WHERE e_mean IS NOT NULL;
ALTER TABLE public.fact_theme_year_aggregates ENABLE ROW LEVEL SECURITY;
DROP POLICY IF EXISTS theme_year_read_all ON public.fact_theme_year_aggregates;
CREATE POLICY theme_year_read_all ON public.fact_theme_year_aggregates
  FOR SELECT TO authenticated USING (true);
DROP POLICY IF EXISTS theme_year_write_service ON public.fact_theme_year_aggregates;
CREATE POLICY theme_year_write_service ON public.fact_theme_year_aggregates
  FOR ALL TO service_role USING (true) WITH CHECK (true);

CREATE TABLE IF NOT EXISTS public.fact_gap_matrix (
  id                            bigserial PRIMARY KEY,
  matrix_id                     text NOT NULL,
  axis_x                        text NOT NULL,
  axis_y                        text NOT NULL,
  depth                         int NOT NULL,
  depth_norm                    real,
  neighbor                      real,
  e_value                       double precision,
  feasibility                   real DEFAULT 0.5,
  publishability                real,
  gap_value                     double precision NOT NULL,
  matrix_confidence_prior       real DEFAULT 0.5,
  matrix_confidence_calibrated  real,
  CHECK (matrix_id IN ('M1','M7','M8')),
  CHECK (depth >= 5)
);
CREATE INDEX IF NOT EXISTS idx_gap_matrix ON public.fact_gap_matrix(matrix_id);
CREATE INDEX IF NOT EXISTS idx_gap_pair   ON public.fact_gap_matrix(matrix_id, axis_x, axis_y);
CREATE INDEX IF NOT EXISTS idx_gap_value  ON public.fact_gap_matrix(gap_value DESC);
CREATE INDEX IF NOT EXISTS idx_gap_axis_x ON public.fact_gap_matrix(axis_x);
CREATE INDEX IF NOT EXISTS idx_gap_axis_y ON public.fact_gap_matrix(axis_y);
ALTER TABLE public.fact_gap_matrix ENABLE ROW LEVEL SECURITY;
DROP POLICY IF EXISTS gap_read_all ON public.fact_gap_matrix;
CREATE POLICY gap_read_all ON public.fact_gap_matrix
  FOR SELECT TO authenticated USING (true);
DROP POLICY IF EXISTS gap_write_service ON public.fact_gap_matrix;
CREATE POLICY gap_write_service ON public.fact_gap_matrix
  FOR ALL TO service_role USING (true) WITH CHECK (true);

INSERT INTO public.schema_migrations (version, description)
VALUES ('0002_static_facts', 'pgvector + dim_theme_embedding (4516) + fact_theme_year_aggregates (53.9K) + fact_gap_matrix (504K)')
ON CONFLICT (version) DO NOTHING;
"""

with psycopg2.connect(DB_URL) as conn:
    with conn.cursor() as cur:
        cur.execute(MIGRATION_0002)
    conn.commit()
print('✓ 0002 migration uygulandı')

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute("SELECT version, description, applied_at FROM public.schema_migrations ORDER BY applied_at;")
    for row in cur.fetchall():
        print(' ', row)

## Cell 4 — Upload `dim_theme` (4516, parent FK)

**Mapping (warehouse → Supabase 0001 schema):**
- `theme_id` → `theme_id`
- `label_en` → `name_en`, `name_tr` (TR çeviri yok, EN kopyalanır — placeholder)
- `theme_id.lower()` → `slug`
- `parent_theme_id` = NULL (hiyerarşi metadata'da; 0001 self-reference yapı subfield→theme FK desteklemiyor)
- `level` = 1
- `metadata` = jsonb {description, keywords, subfield_id/name, field_id/name, domain_id/name, works_count}

In [ ]:
df = pd.read_parquet(PATHS['dim_theme'])
print(f'rows: {len(df):,}  cols: {list(df.columns)}')

import re
def slugify(s):
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9]+', '-', s).strip('-')
    return s[:255] if s else 'theme'

META_KEYS = ['description', 'keywords', 'subfield_id', 'subfield_name',
             'field_id', 'field_name', 'domain_id', 'domain_name', 'works_count']

rows = []
seen_slugs = set()
for r in df.to_dict('records'):
    tid = str(r['theme_id'])
    label = r.get('label_en') or tid
    slug = slugify(tid)
    while slug in seen_slugs:
        slug = slug + '-x'
    seen_slugs.add(slug)
    meta = {}
    for k in META_KEYS:
        v = r.get(k)
        if v is None or (isinstance(v, float) and math.isnan(v)):
            continue
        if hasattr(v, 'tolist'):
            v = v.tolist()
        meta[k] = v
    rows.append((tid, label, label, slug, None, 1, json.dumps(meta, ensure_ascii=False)))

sql = """
INSERT INTO public.dim_theme (theme_id, name_tr, name_en, slug, parent_theme_id, level, metadata)
VALUES %s
ON CONFLICT (theme_id) DO NOTHING;
"""

t0 = time.time()
with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    for i in range(0, len(rows), CHUNK):
        execute_values(cur, sql, rows[i:i+CHUNK], page_size=5000)
        print(f'  chunk {i//CHUNK + 1}: {min(i+CHUNK, len(rows)):,}/{len(rows):,}')
    conn.commit()
print(f'✓ dim_theme uploaded in {time.time()-t0:.1f}s')

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM public.dim_theme;')
    print(f'  DB count: {cur.fetchone()[0]:,}')

## Cell 5 — Upload `dim_theme_embedding` (4516 × 256-d, pgvector)

**Mapping:** birebir aynalama (theme_id, embedding, text_source, n_terms_used).

**Vector format:** psycopg2 binary protokolü pgvector'ı desteklemediği için text-cast `'[0.1,0.2,...]'` formatı kullanılır.

In [ ]:
df = pd.read_parquet(PATHS['dim_theme_embedding'])
print(f'rows: {len(df):,}  cols: {list(df.columns)}')

def vec_to_pg(v):
    arr = list(v) if not isinstance(v, list) else v
    assert len(arr) == 256, f'embedding boyutu 256 değil: {len(arr)}'
    return '[' + ','.join(f'{float(x):.6f}' for x in arr) + ']'

rows = []
for r in df.to_dict('records'):
    tid = str(r['theme_id'])
    emb = vec_to_pg(r['embedding'])
    text_source = r.get('text_source')
    n_terms = r.get('n_terms_used')
    if isinstance(n_terms, float) and math.isnan(n_terms):
        n_terms = None
    elif n_terms is not None:
        n_terms = int(n_terms)
    rows.append((tid, emb, text_source, n_terms))

sql = """
INSERT INTO public.dim_theme_embedding (theme_id, embedding, text_source, n_terms_used)
VALUES %s
ON CONFLICT (theme_id) DO NOTHING;
"""

t0 = time.time()
with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    for i in range(0, len(rows), CHUNK):
        execute_values(cur, sql, rows[i:i+CHUNK], page_size=2500)
        print(f'  chunk {i//CHUNK + 1}: {min(i+CHUNK, len(rows)):,}/{len(rows):,}')
    conn.commit()
print(f'✓ dim_theme_embedding uploaded in {time.time()-t0:.1f}s')

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM public.dim_theme_embedding;')
    print(f'  DB count: {cur.fetchone()[0]:,}')

## Cell 6 — Upload `fact_theme_year_aggregates` (53,943 × 15, t-ESTRA 7/8)

**Mapping:** parquet kolonları lowercase rename — birebir aynalama.
- R_mean → r_mean, E_mean → e_mean, ED_k_simpson → ed_k_simpson, ES_k_top10 → es_k_top10
- TS_penalty → ts_penalty, C_k → c_k, CD_k → cd_k, MQ_k → mq_k
- blocked_metrics List[String] → text[] (Postgres array)
- n_blocked → n_blocked

In [ ]:
df = pd.read_parquet(PATHS['fact_theme_year_aggregates'])
print(f'rows: {len(df):,}  cols: {list(df.columns)}')

def fnum(v):
    if v is None: return None
    try:
        f = float(v)
        return None if math.isnan(f) else f
    except (TypeError, ValueError):
        return None

def iint(v):
    if v is None: return None
    try:
        f = float(v)
        return None if math.isnan(f) else int(f)
    except (TypeError, ValueError):
        return None


def norm_theme_id(v):
    """dim_theme uses 'T10001' format; this parquet may store '10001' or int.
    Force 'T' prefix to satisfy FK to dim_theme."""
    s = str(v).strip()
    if s.startswith('T'):
        return s
    try:
        return f'T{int(float(s))}'
    except (ValueError, TypeError):
        return f'T{s}'

rows = []
for r in df.to_dict('records'):
    bm = r.get('blocked_metrics')
    if bm is None:
        bm_list = []
    elif hasattr(bm, 'tolist'):
        bm_list = list(bm.tolist())
    else:
        bm_list = list(bm)
    bm_list = [str(x) for x in bm_list]

    rows.append((
        norm_theme_id(r['theme_id']),
        int(r['year']),
        iint(r.get('paper_count')) or 0,
        fnum(r.get('R_mean')),
        fnum(r.get('R_std')),
        fnum(r.get('E_mean')),
        fnum(r.get('E_std')),
        fnum(r.get('ED_k_simpson')),
        fnum(r.get('ES_k_top10')),
        fnum(r.get('TS_penalty')),
        fnum(r.get('C_k')),
        fnum(r.get('CD_k')),
        fnum(r.get('MQ_k')),
        bm_list,
        iint(r.get('n_blocked')) or 0,
    ))

sql = """
INSERT INTO public.fact_theme_year_aggregates
  (theme_id, year, paper_count, r_mean, r_std, e_mean, e_std,
   ed_k_simpson, es_k_top10, ts_penalty, c_k, cd_k, mq_k,
   blocked_metrics, n_blocked)
VALUES %s
ON CONFLICT (theme_id, year) DO NOTHING;
"""

t0 = time.time()
with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    for i in range(0, len(rows), CHUNK):
        execute_values(cur, sql, rows[i:i+CHUNK], page_size=5000)
        if (i // CHUNK) % 2 == 0:
            print(f'  chunk {i//CHUNK + 1}: {min(i+CHUNK, len(rows)):,}/{len(rows):,}')
    conn.commit()
print(f'✓ fact_theme_year_aggregates uploaded in {time.time()-t0:.1f}s')

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM public.fact_theme_year_aggregates;')
    print(f'  DB count: {cur.fetchone()[0]:,}')

## Cell 7 — Upload `fact_gap_matrix` (504,436 × 12, M1+M7+M8)

**Mapping:** birebir + tek rename `E` → `e_value` (Postgres reserved word çakışması yok ama kolon adı disiplini için lowercase + alfasayısal).

**Süre tahmini:** 504K satır × ~50 chunk × ~3-5s = 3-5 dakika.

In [ ]:
df = pd.read_parquet(PATHS['fact_gap_matrix'])
print(f'rows: {len(df):,}  cols: {list(df.columns)}')
print(f'matrix_id distribution:\n{df["matrix_id"].value_counts()}')

rows = []
for r in df.to_dict('records'):
    rows.append((
        str(r['matrix_id']),
        str(r['axis_x']),
        str(r['axis_y']),
        int(r['depth']),
        fnum(r.get('depth_norm')),
        fnum(r.get('neighbor')),
        fnum(r.get('E')),
        fnum(r.get('feasibility')),
        fnum(r.get('publishability')),
        fnum(r.get('gap_value')),
        fnum(r.get('matrix_confidence_prior')),
        fnum(r.get('matrix_confidence_calibrated')),
    ))

sql = """
INSERT INTO public.fact_gap_matrix
  (matrix_id, axis_x, axis_y, depth, depth_norm, neighbor, e_value,
   feasibility, publishability, gap_value,
   matrix_confidence_prior, matrix_confidence_calibrated)
VALUES %s;
"""

t0 = time.time()
with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    for i in range(0, len(rows), CHUNK):
        execute_values(cur, sql, rows[i:i+CHUNK], page_size=5000)
        if (i // CHUNK) % 5 == 0:
            elapsed = time.time() - t0
            done = min(i + CHUNK, len(rows))
            rate = done / elapsed if elapsed > 0 else 0
            print(f'  chunk {i//CHUNK + 1}: {done:,}/{len(rows):,}  ({rate:,.0f} rows/s)')
    conn.commit()
print(f'✓ fact_gap_matrix uploaded in {time.time()-t0:.1f}s')

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM public.fact_gap_matrix;')
    print(f'  DB count: {cur.fetchone()[0]:,}')
    cur.execute('SELECT matrix_id, COUNT(*) FROM public.fact_gap_matrix GROUP BY matrix_id ORDER BY matrix_id;')
    for row in cur.fetchall():
        print(f'    {row[0]}: {row[1]:,}')

## Cell 8 — Verify (row counts + pgvector cosine smoke)

**Beklenen:**
- dim_theme = 4,516
- dim_theme_embedding = 4,516
- fact_theme_year_aggregates = 53,943
- fact_gap_matrix = 504,436
- pgvector smoke: 1 random theme'e top-5 cosine komşu, similarity ∈ (0, 1]

In [ ]:
EXPECTED = {
    'dim_theme': 4516,
    'dim_theme_embedding': 4516,
    'fact_theme_year_aggregates': 53943,
    'fact_gap_matrix': 504436,
}

with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    print('Row counts:')
    for tbl, exp in EXPECTED.items():
        cur.execute(f'SELECT COUNT(*) FROM public.{tbl};')
        got = cur.fetchone()[0]
        flag = '✓' if got == exp else ('~' if abs(got - exp) < exp * 0.01 else '✗')
        print(f'  {flag} {tbl:32s} = {got:>10,}  (expected {exp:,})')

    print('\npgvector cosine smoke (1 random theme → top-5 neighbors):')
    cur.execute("""
        WITH q AS (SELECT theme_id, embedding FROM public.dim_theme_embedding ORDER BY random() LIMIT 1)
        SELECT q.theme_id AS query_theme,
               t.theme_id AS neighbor_theme,
               1 - (t.embedding <=> q.embedding) AS cosine_sim
        FROM public.dim_theme_embedding t, q
        WHERE t.theme_id != q.theme_id
        ORDER BY t.embedding <=> q.embedding
        LIMIT 5;
    """)
    for row in cur.fetchall():
        print(f'  {row[0]} → {row[1]}  cos={row[2]:.4f}')

    print('\nfact_gap_matrix top-5 by gap_value:')
    cur.execute("""
        SELECT matrix_id, axis_x, axis_y, depth, gap_value
        FROM public.fact_gap_matrix
        ORDER BY gap_value DESC LIMIT 5;
    """)
    for row in cur.fetchall():
        print(f'  {row}')

print('\n✓ Yükleme tamam — Supabase static fact stack hazır.')